In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# 1. Wczytanie i przygotowanie danych
df = pd.read_csv("cases_clinical_for_lab12.csv")

# Mapowanie zmiennych kategorialnych na wartości 0/1
df["sex"] = df["sex"].map({"M": 1, "F": 0})
df["smoker"] = df["smoker"].map({"yes": 1, "no": 0})
df["family_history"] = df["family_history"].map({"yes": 1, "no": 0})

# Usunięcie ewentualnych braków danych
df = df.dropna()

X = df.drop("high_risk_cvd", axis=1)
y = df["high_risk_cvd"]

# Podział na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 2. Trenowanie modelu Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Pobranie prawdopodobieństw
y_probs = rf.predict_proba(X_test)[:, 1]

# 3. Definicja kosztów i przeszukiwanie progów
C_FN = 10
C_FP = 1
thresholds = np.arange(0.1, 0.91, 0.01)
results = []

for tau in thresholds:
    y_pred = (y_probs >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    cost = (C_FN * fn) + (C_FP * fp)
    results.append((tau, cost, fn, fp))

# 4. Znalezienie progu minimalizującego koszt
df_results = pd.DataFrame(results, columns=["Threshold", "Cost", "FN", "FP"])
optimal = df_results.loc[df_results["Cost"].idxmin()]

print(f"Optymalny próg (tau): {optimal['Threshold']:.2f}")
print(f"Minimalny koszt całkowity: {optimal['Cost']:.0f}")
print(f"Błędy przy tym progu -> FN: {optimal['FN']:.0f}, FP: {optimal['FP']:.0f}")


Optymalny próg (tau): 0.18
Minimalny koszt całkowity: 90
Błędy przy tym progu -> FN: 0, FP: 90
